# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamzaviour/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

Lane: Content Opportunity Scoring. This notebook (1) reads two findings from FlyRank's March 2026 paper the way I would want my own work read, then (2) attacks my Week-5 model. Language is observed / measured / directional / decision-support throughout.

## 1. Two paper findings + my methodology questions

The paper is explicit that it is observational, that ML pages are exploratory, and that it prefers direct comparisons. That openness is what makes specific questions possible.

### Finding A — "Finding #1: The Anatomy of Growing Content" (CONFIRMED)
*Claim as I read it:* growing pages (74.8K) are longer (3.2K vs 2.3K words) and younger (184d vs 230d) than declining pages (45.6K).

**My questions (constructive):**
1. **Where does the label come from?** "Up" and "down" come from the 30-day vs previous-30-day impression change (paper thresholds ±10%). The page-profile columns (age, position, health) are measured over a window that includes those same 60 days. Were the profile columns measured before the trend window, or alongside it? If alongside, a page that just surged will also look "better positioned", so some of the profile gap may describe the label rather than precede it.
2. **Is the age gap partly a definition effect?** The paper notes "New" pages (created within 30 days) are their own trend class, and young pages have little previous-30-day base, so small absolute changes become large percentage changes. How are young pages with a tiny prior base counted in "up"?
3. **Which threshold is used?** The paper states ±10%, while the starter release documents ±20% for the same `trend_direction`. A short note on which version produced the 74.8K/45.6K counts would let a reader reproduce them.
4. **Does the gap hold within groups?** Because the study is observational, would the word-count gap survive when comparing pages of similar age or from the same brand? The paper already says content age confounds comparisons; a within-age-band view would show how much is age.

### Finding B — ML appendix, "What Predicts Growth?" (logistic regression, 71% holdout accuracy)
**My questions (constructive):**
1. **Does the validation design support "holdout"?** The pipeline lists an 80/20 split. If pages from the same brand sit on both sides, the model can recognise a brand, not generalise to a new one. A split grouped by brand would show whether 71% survives.
2. **What is 71% compared with?** Declining pages are about 38% of up+down pages here (45.6K of 120.4K), so a majority-class guess already scores about 62%. Accuracy next to the base rate (or AUC / precision@K) would say how much skill 71% is.
3. **Are any inputs inside the label's window?** "Days visible" and impressions are named as strong positive signals. If they are counted over the same period as the 30-day change, they may partly encode the outcome. The paper's own note that the health-score importances are partly built-in shows the authors already watch for this; the same check on the growth model would be informative.

I am not grading the paper: it discloses its limits and calls the ML pages descriptive. I am applying the same questions to my own model below.

In [1]:
import os, json, warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
if os.path.exists("../../data/raw/content_refresh_anonymized.csv"): os.chdir("../..")
os.makedirs("work/outputs", exist_ok=True)
PATH = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(PATH):
    import urllib.request
    PATH = "content_refresh_anonymized.csv"
    urllib.request.urlretrieve("https://raw.githubusercontent.com/Hamzaviour/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv", PATH)
df = pd.read_csv(PATH)
df["declining"] = (df.trend_direction == "down").astype(int)
y, groups, SEED = df.declining.values, df.client_id.values, 42
print(df.shape, "base rate declining:", round(y.mean(), 3))

(30000, 45) base rate declining: 0.542


## 2. My Week-5 model under an honest split (before / after)

Week 5 already used a client-grouped split, so the "before" here is the split a careless version would use: a **random** 5-fold split (pages from the same client on both sides). The "after" is the honest **client-grouped** 5-fold split. Same model, same features (the Week-5 set), same seed. The gap is a measure of how much client memorisation a random split would have rewarded.

In [2]:
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

def build_X(num, cat=("content_type", "main_intent", "competition_level"), logs=(), flags=()):
    X = df[list(num) + list(cat)].copy()
    for c in flags: X["has_" + c] = X[c].notna().astype(int)
    if "avg_position" in num: X["pos_missing"] = (X.avg_position == 0).astype(int)
    for c in logs: X[c] = np.log1p(X[c].clip(lower=0))
    return pd.get_dummies(X, columns=list(cat), dummy_na=True)

W5_NUM = ["search_volume", "competition", "cpc", "word_count", "char_count", "impressions_90d", "clicks_90d",
          "sessions_90d", "ai_sessions_90d", "days_with_impressions", "days_with_sessions", "content_age_days",
          "days_since_last_update", "ctr", "avg_position", "engagement_rate", "scroll_rate"]
X_w5 = build_X(W5_NUM, logs=["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"],
               flags=["search_volume", "word_count", "engagement_rate", "scroll_rate"])

MODELS = {
    "logistic regression": lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000, C=0.5)),
    "random forest": lambda: make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, min_samples_leaf=20, n_jobs=-1, random_state=SEED)),
}
def prec_at_k(s, l, k): return l[np.argsort(-s, kind="stable")[:k]].mean()
def evaluate(X, folds, make):
    aucs, p50, p200 = [], [], []
    for tr, te in folds:
        m = make().fit(X.iloc[tr], y[tr]); p = m.predict_proba(X.iloc[te])[:, 1]
        aucs.append(roc_auc_score(y[te], p)); p50.append(prec_at_k(p, y[te], 50)); p200.append(prec_at_k(p, y[te], 200))
    return {"AUC": np.mean(aucs), "AUC_sd": np.std(aucs), "P@50": np.mean(p50), "P@200": np.mean(p200)}

rand_folds  = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(X_w5, y))
group_folds = list(GroupKFold(5).split(X_w5, y, groups))
rows = {}
for name, mk in MODELS.items():
    rows[f"{name} | random split (before)"]  = evaluate(X_w5, rand_folds, mk)
    rows[f"{name} | client-grouped (after)"]  = evaluate(X_w5, group_folds, mk)
t2 = pd.DataFrame(rows).T
t2.loc["(base rate / random picking)"] = [0.5, np.nan, y.mean(), y.mean()]
print(t2.round(3).to_string())

                                                AUC  AUC_sd   P@50  P@200
logistic regression | random split (before)   0.715   0.006  0.884  0.857
logistic regression | client-grouped (after)  0.656   0.034  0.796  0.783
random forest | random split (before)         0.773   0.005  0.972  0.937
random forest | client-grouped (after)        0.673   0.038  0.792  0.757
(base rate / random picking)                  0.500     NaN  0.542  0.542


**Reading:** the random split scores higher than the client-grouped split for the same model. The size of that gap (see table) is a measured estimate of how much hidden client character the model can use when it has seen the client before. Only the grouped number answers "does it work on a client it has never seen?", so that is the number I keep.

## 3. Leakage audit

Three checks from the attack list, each with a number.

**3a. Deliberate leak (does the harness catch it?).** Add `trend_pct` (the label's source) and confirm the score jumps; then remove it.

In [3]:
X_leak = X_w5.copy(); X_leak["LEAK_trend_pct"] = df.trend_pct.fillna(0).values
leak = evaluate(X_leak, group_folds, MODELS["logistic regression"])
honest = evaluate(X_w5, group_folds, MODELS["logistic regression"])
print(f"WITH trend_pct leak: AUC {leak['AUC']:.3f}   <- jumps toward 1.0, the harness works")
print(f"WITHOUT (Week-5 set): AUC {honest['AUC']:.3f}   <- kept")
del X_leak

WITH trend_pct leak: AUC 0.998   <- jumps toward 1.0, the harness works
WITHOUT (Week-5 set): AUC 0.656   <- kept


**3b. Timeline / window overlap (the sneaky one).** The label compares the last 30 days with the 30 days before. Every Week-5 feature measured over the 90-day window (`impressions_90d`, `clicks_90d`, `ctr`, `avg_position`, `days_with_impressions`, `engagement_rate`, …) *contains* those 60 label days, so none of them is strictly "knowable before" the label. The only 90-day-window quantity with a clean sub-window is volume: `total_90d − last_30d − prev_30d` is the **earliest 30 days** (days 61-90 back), which precedes the label window. Static metadata (word count, content type, intent, search volume, age, days since update) is not window-dependent.

I therefore build a **strict-timeline** feature set: static metadata + earliest-30-day volumes, and compare it with the Week-5 set on the same grouped folds.

In [4]:
for c in ["impressions", "clicks", "sessions"]:
    df[f"{c}_earliest30"] = (df[f"{c}_90d"] - df[f"{c}_last_30d"] - df[f"{c}_prev_30d"]).clip(lower=0)
print("share of pages where 90d total < last30+prev30 (rounding / registration artefacts):",
      round(((df.impressions_90d - df.impressions_last_30d - df.impressions_prev_30d) < 0).mean(), 4))

STRICT_NUM = ["search_volume", "competition", "cpc", "word_count", "char_count", "content_age_days", "days_since_last_update",
              "impressions_earliest30", "clicks_earliest30", "sessions_earliest30"]
X_strict = build_X(STRICT_NUM, logs=["impressions_earliest30", "clicks_earliest30", "sessions_earliest30"],
                   flags=["search_volume", "word_count"])
rows3 = {}
for name, mk in MODELS.items():
    rows3[f"{name} | Week-5 features (overlap)"] = evaluate(X_w5, group_folds, mk)
    rows3[f"{name} | strict timeline"]           = evaluate(X_strict, group_folds, mk)
t3 = pd.DataFrame(rows3).T
t3.loc["(base rate / random picking)"] = [0.5, np.nan, y.mean(), y.mean()]
print(t3.round(3).to_string())

share of pages where 90d total < last30+prev30 (rounding / registration artefacts): 0.0


                                                   AUC  AUC_sd   P@50  P@200
logistic regression | Week-5 features (overlap)  0.656   0.034  0.796  0.783
logistic regression | strict timeline            0.665   0.065  0.828  0.776
random forest | Week-5 features (overlap)        0.673   0.038  0.792  0.757
random forest | strict timeline                  0.622   0.062  0.712  0.695
(base rate / random picking)                     0.500     NaN  0.542  0.542


**3c. Other checks.** Population filter: the starter slice keeps only pages with at least 1 impression in the 90 days and age ≥ 90 days, so it is an *active-pages* population, which I disclose as a limitation (the paper makes the same kind of choice for its ML subset). No product flags (`Fix CTR`, health score, etc.) are columns here and none are used. IDs are used only for grouping. `provider_used` / `model_used` are excluded.

In [5]:
audit = pd.DataFrame([
    ("label-derived columns (trend_*)",            "excluded", "train-with/without test above: jump is large, so they stay out"),
    ("30-day windows (last_30d / prev_30d)",       "excluded from features", "label is computed from them"),
    ("90-day totals, ctr, position, engagement",   "OVERLAP with label window", "kept in Week-5 set; dropped in strict set"),
    ("static metadata (word_count, type, age ...)","ok",        "not window-dependent"),
    ("earliest-30-day volumes",                    "ok",        "precede the label window"),
    ("product flags / health score",               "not present","would be baseline-only if available"),
    ("IDs",                                        "grouping only", "GroupKFold by client"),
], columns=["input", "status", "note"])
print(audit.to_string(index=False))
json.dump({"split_before_after": t2.round(4).to_dict("index"), "leakage_timeline": t3.round(4).to_dict("index"),
           "deliberate_leak_auc": round(leak["AUC"], 4)}, open("work/outputs/w06_validation_audit_metrics.json", "w"), indent=2)

                                      input                    status                                                           note
            label-derived columns (trend_*)                  excluded train-with/without test above: jump is large, so they stay out
       30-day windows (last_30d / prev_30d)    excluded from features                                    label is computed from them
   90-day totals, ctr, position, engagement OVERLAP with label window                      kept in Week-5 set; dropped in strict set
static metadata (word_count, type, age ...)                        ok                                           not window-dependent
                    earliest-30-day volumes                        ok                                       precede the label window
               product flags / health score               not present                            would be baseline-only if available
                                        IDs             grouping only

### Real failure examples (strict-timeline logistic model, held-out clients)

In [6]:
oof = np.zeros(len(df))
for tr, te in group_folds:
    m = MODELS["logistic regression"]().fit(X_strict.iloc[tr], y[tr]); oof[te] = m.predict_proba(X_strict.iloc[te])[:, 1]
df["p_decl"] = oof
cols = ["content_id", "p_decl", "declining", "content_type", "content_age_days", "impressions_earliest30", "impressions_prev_30d", "impressions_last_30d", "trend_pct"]
print("Confident false alarms (predicted declining, actually not):")
print(df[df.declining == 0].nlargest(3, "p_decl")[cols].round(2).to_string(index=False))
print("\nConfident misses (predicted safe, actually declining):")
print(df[df.declining == 1].nsmallest(3, "p_decl")[cols].round(2).to_string(index=False))
small = df[df.impressions_prev_30d < 50]
print(f"\nPages with <50 impressions in the prior 30 days: n={len(small):,}, declining share {small.declining.mean():.2f} vs {df.declining.mean():.2f} overall")

Confident false alarms (predicted declining, actually not):
          content_id  p_decl  declining    content_type  content_age_days  impressions_earliest30  impressions_prev_30d  impressions_last_30d  trend_pct
content_b2ec40b9d68c    0.94          0 keyword article               139                    5654                  6947                 10856       56.3
content_013a5d3dfc26    0.93          0 keyword article               148                   20175                 21793                 23684        8.7
content_919827b5860a    0.93          0 keyword article               229                    9061                 13416                 24741       84.4

Confident misses (predicted safe, actually declining):
          content_id  p_decl  declining    content_type  content_age_days  impressions_earliest30  impressions_prev_30d  impressions_last_30d  trend_pct
content_a0a76e94ade5     0.0          1 keyword article               175                       8                    10

## 4. Claim rewrite

What the audit measured (seed 42, 5 folds, client-held-out unless stated):

- **Split.** A random split would have reported AUC 0.72 (logistic) and 0.77 (forest) with P@50 of 0.88 and 0.97. Under the client-grouped split the same models score AUC 0.66 and 0.67 with P@50 of about 0.80. The random-split numbers are **inflated by about 0.06-0.10 AUC** because the model has seen the same client's pages in training.
- **Leak harness.** Adding `trend_pct` pushes AUC to 0.998, so the harness does detect a leak; the Week-5 number (0.656) contains no such column.
- **Window overlap.** Replacing the overlapping 90-day features with a strict-timeline set (static metadata + the earliest 30 days of volume) left logistic regression about the same (AUC 0.665 vs 0.656, within fold-to-fold spread of 0.03-0.07) and lowered the forest (0.622 vs 0.673). So the directional signal does not appear to depend on the overlapping columns for the logistic model, but the forest's extra edge in Week 5 was partly using them.
- **Failure shape.** Confident misses are pages with tiny prior-30-day volume (1 to 10 impressions) where a small absolute change is a large percentage drop; confident false alarms are large pages that happened to grow. Both are close to what the earliest-30-day volume alone would suggest, i.e. a regression-to-the-mean pattern.

| Week-5 claim | Problem | Rewritten claim |
|---|---|---|
| "The models beat the baseline and the base rate." | True on the grouped split, but the margin is modest and fold spread is 0.03-0.07 AUC. | In this snapshot, client-held-out logistic regression scored a measured AUC of about 0.66 versus 0.5 for random ordering; this is a directional signal, not a reliable forecast. |
| "Random forest wins on AUC (0.67)." | The forest's edge shrinks or reverses under a strict timeline, and the gap to logistic is inside fold noise. | Logistic regression and the forest performed similarly (AUC 0.62-0.67 depending on feature set); I prefer the logistic model because it is simpler and did not lose skill under the stricter timeline. |
| "Precision@50 of 0.80 for logistic." | Base rate is 0.54, folds are small and differ in base rate (0.38-0.65). | Among the top 50 pages per held-out client fold, about 0.80 were labelled declining (base rate 0.54); differences under about 0.03 should not be read as real. |
| "It leans on average position and content age." | Position is a 90-day figure that overlaps the label window, and position `0` means "no data" (a coverage effect). | Position and age were the most influential inputs in the Week-5 set; part of that is measurement coverage and window overlap, so it is descriptive, not causal. |
| "Pages flagged are the ones to review." | Label is a 30-day impression drop, which includes mean-reversion on tiny-volume pages. | The score is decision-support for ordering a review queue: a flag means "worth a human look", not "will decline". |

**What I would still not claim:** that any feature *causes* decline, that the result transfers to the warehouse or to future months (this is one snapshot, one active-pages population with age ≥ 90 days and at least one impression), or that the model beats a time-aware split I have not run. The next step with the warehouse daily table is a time-based split (train on earlier months, test on a later one).

## 5. Self-check

- [x] Two paper findings named, each with constructive methodology questions (section 1)
- [x] Week-5 model re-run under random vs client-grouped split with before/after table (section 2)
- [x] Leakage audit: deliberate leak, window-overlap check with a strict-timeline re-run, status table (section 3)
- [x] Real failure examples shown (section 3)
- [x] Claims rewritten with safe language (section 4)
- [x] No client names or URLs; seeds fixed (42); metrics JSON in `work/outputs/`